# Jukebox Audio Prompt Quickstart

This notebook is focused on one thing only: upload your own audio and have Jukebox continue from it using the `1b_lyrics` model.

Important limitation: this notebook only generates the top level, so the output will sound noisy compared with a fully upsampled Jukebox sample. That tradeoff keeps the notebook much more usable in Colab.


## Step 1: Install The Fork

What this step does:
- installs Jukebox directly from `emcee3/jukebox`
- ensures people opening your fork get your Colab-compatible version

What you can modify:
- usually nothing
- only change `REPO_URL` if you want to test a different repository or branch


In [0]:
REPO_URL = "https://github.com/emcee3/jukebox.git"
!pip install git+{REPO_URL}

## Step 2: Check The GPU Runtime

What this step does:
- confirms Colab assigned a GPU runtime
- helps catch setup issues before model loading starts

What you can modify:
- nothing in the cell itself
- if it fails, change the Colab runtime type to GPU and rerun


In [0]:
!nvidia-smi

## Step 3: Import Helpers And Pick The Device

What this step does:
- imports the notebook dependencies
- sets up a normal single-GPU Colab device

What you can modify:
- usually nothing
- advanced users can import extra helpers here if they want to experiment


In [0]:
import torch as t
from IPython.display import Audio
from google.colab import files
from jukebox.make_models import make_prior, make_vqvae, MODELS
from jukebox.hparams import Hyperparams, setup_hparams
from jukebox.sample import _sample, load_prompts
from jukebox.utils.dist_utils import setup_dist_from_mpi

rank, local_rank, device = setup_dist_from_mpi()
device

## Step 4: Load The Jukebox Model

What this step does:
- loads the `1b_lyrics` model, which is the most practical Jukebox lyrics model for Colab
- configures the timing values Jukebox expects for top-level generation

What you can modify:
- `hps.n_samples`: keep this at `1` unless you are deliberately testing memory limits
- `sample_window_length`: keep the default unless you know the model timing constraints
- `minimum_total_length`: keep the default unless you understand the label timing range


In [0]:
model = "1b_lyrics"
hps = Hyperparams()
hps.sr = 44100
hps.n_samples = 1
hps.levels = 3
hps.hop_fraction = [0.5, 0.5, 0.125]
sample_window_length = 786432
minimum_total_length = 786816

vqvae_name, *prior_names = MODELS[model]
vqvae_hps = setup_hparams(vqvae_name, dict(sample_length=sample_window_length))
vqvae = make_vqvae(vqvae_hps, device)
top_prior = make_prior(setup_hparams(prior_names[-1], dict()), vqvae, device)
hps.sample_length = sample_window_length
hps.total_length = minimum_total_length
dict(sample_seconds=hps.sample_length / hps.sr, total_seconds=hps.total_length / hps.sr, raw_to_tokens=top_prior.raw_to_tokens)

## Step 5: Choose The Conditioning Labels

What this step does:
- sets the artist, genre, and lyrics metadata used while continuing from your uploaded audio
- gives you the main steering controls for how Jukebox interprets the prompt

What you can modify:
- `artist`
- `genre`
- `lyrics`

Tip: even with your own prompt audio, these labels still matter. Changing them can bend the continuation toward different styles or lyrical behavior.


In [0]:
metas = [dict(
    artist="Zac Brown Band",
    genre="Country",
    total_length=hps.total_length,
    offset=0,
    lyrics="""I met a traveller from an antique land,
    Who said Two vast and trunkless legs of stone
    Stand in the desert. Near them, on the sand,
    Half sunk a shattered visage lies, whose frown,
    And wrinkled lip, and sneer of cold command,
    Tell that its sculptor well those passions read
    Which yet survive, stamped on these lifeless things,
    The hand that mocked them, and the heart that fed;
    And on the pedestal, these words appear:
    My name is Ozymandias, King of Kings;
    Look on my Works, ye Mighty, and despair!
    Nothing beside remains. Round the decay
    Of that colossal Wreck, boundless and bare
    The lone and level sands stretch far away
    """
)] * hps.n_samples
labels = [None, None, top_prior.labeller.get_batch_labels(metas, 'cuda')]
labels[-1]['y'].shape

## Step 6: Define The Sampling Helpers

What this step does:
- creates small helper functions used by the final generation step
- keeps the actual sampling cell shorter and easier to modify

What you can modify:
- `temp` inside `top_level_sampling_kwargs()` if you want a different default temperature
- advanced users can change chunking or batch size here, but the defaults are the safest place to start


In [0]:
def make_labels():
    return [None, None, top_prior.labeller.get_batch_labels(metas, 'cuda')]

def top_level_sampling_kwargs(temp=0.98):
    return [
        dict(temp=0.99, fp16=True, max_batch_size=16, chunk_size=32),
        dict(temp=0.99, fp16=True, max_batch_size=16, chunk_size=32),
        dict(temp=temp, fp16=True, max_batch_size=16, chunk_size=32),
    ]

def run_top_level(zs, output_name, temp=0.98):
    hps.name = output_name
    return _sample(zs, make_labels(), top_level_sampling_kwargs(temp), [None, None, top_prior], [2], hps)


## Step 7: Upload Your Audio Prompt

What this step does:
- uploads your local audio file into the Colab session
- stores its path so later cells can encode and continue from it

What you can modify:
- nothing in the code cell
- choose a different audio file when the upload dialog opens

Good prompt choices are usually short WAV files with a clear idea in the first few seconds.


In [0]:
uploaded = files.upload()
uploaded_audio_path = next(iter(uploaded)) if uploaded else None
uploaded_audio_path

## Step 8: Listen To The Uploaded Prompt

What this step does:
- plays back the uploaded file inside the notebook
- helps you confirm you uploaded the correct clip before generation starts

What you can modify:
- nothing in the code
- rerun the upload step with a different file if needed


In [0]:
assert uploaded_audio_path is not None, 'Upload an audio file first.'
Audio(uploaded_audio_path)

## Step 9: Continue From The Uploaded Audio

What this step does:
- takes the first part of your uploaded clip as a prompt
- encodes it into Jukebox tokens
- asks the top prior to continue from that prompt
- saves the generated audio under `prompted_samples/level_2/item_0.wav`

What you can modify:
- `prompt_length_in_seconds`: how much of your uploaded clip is used as context
- `sampling_temperature`: higher means more randomness, lower means more conservative continuation

Tip: start with prompt lengths around `4` to `12` seconds. The prompt must stay shorter than the full sample window.


## Runtime Check: Is The Machine Too Weak Or Is The Cell Just Busy?

What this step does:
- shows the GPU name and available VRAM
- explains what symptoms usually mean `too weak`, `out of memory`, or `still running normally`

What you can modify:
- nothing in the code cell
- if generation is too slow or unstable, go back and reduce `prompt_length_in_seconds` or keep `hps.n_samples = 1`

How to interpret it:
- If the generation cell prints sampling progress and the notebook stays responsive, it is usually still working.
- If you get CUDA out-of-memory errors, the runtime is too small for the chosen settings.
- If the GPU has very low memory or a weak model name compared with T4/L4/A100, expect noticeably slower runs.
- If a cell runs for a very long time with no new output and the GPU is barely used, restarting the runtime and trying a shorter prompt is a good next step.


In [0]:
gpu_name = t.cuda.get_device_name(0) if t.cuda.is_available() else 'cpu-only runtime'
total_vram_gb = round(t.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2) if t.cuda.is_available() else 0
dict(gpu=gpu_name, total_vram_gb=total_vram_gb, n_samples=hps.n_samples, sample_seconds=hps.sample_length / hps.sr)

In [0]:
prompt_length_in_seconds = 8
sampling_temperature = 0.98
assert uploaded_audio_path is not None, 'Upload an audio file first.'
prompt_duration = (int(prompt_length_in_seconds * hps.sr) // top_prior.raw_to_tokens) * top_prior.raw_to_tokens
assert 0 < prompt_duration < hps.sample_length, 'Choose a prompt shorter than the full sample window.'
prompt_audio = load_prompts([uploaded_audio_path], prompt_duration, hps)
prompted_zs = top_prior.encode(prompt_audio, start_level=0, end_level=len(prior_names), bs_chunks=prompt_audio.shape[0])
prompted_zs = run_top_level(prompted_zs, output_name='prompted_samples', temp=sampling_temperature)
dict(prompt_seconds=prompt_duration / hps.sr, generated_seconds=hps.sample_length / hps.sr)

## Step 10: Listen To The Result

What this step does:
- plays back the Jukebox continuation generated from your uploaded prompt

What you can modify:
- nothing in this playback cell
- if you want a different result, go back and change the labels, prompt length, or temperature, then rerun the generation step


In [0]:
Audio('prompted_samples/level_2/item_0.wav')

## Main Knobs To Play With

Once the notebook works end to end, the most useful things to change are:
- the uploaded audio clip
- `artist`
- `genre`
- `lyrics`
- `prompt_length_in_seconds`
- `sampling_temperature`

Jukebox is not a precise audio editor, so the best mindset is: upload a prompt, steer the model with labels, and explore different reinterpretations.
